# ILT Validation Notebook: Gaussian vs Non-Gaussian

目标（对应你要的三个实验）：
1. Gaussian clean signal → ILT（验证 pipeline 正常，能得到 sharp peak）
2. Non-Gaussian clean signal + sweep sphere radius（看 peak shift / broadening）
3. Non-Gaussian noisy signal + sweep SNR & alpha（看 DEI / pathway error 是否变差）


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
repo_root = None
for c in candidate_roots:
    if (c / "dexsy_core").exists():
        repo_root = c
        break
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing dexsy_core.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dexsy_core import (
    ForwardModel3CNonGaussian,
    create_forward_model,
    compute_dei,
)

np.set_printoptions(precision=4, suppress=True)
print("Repo root:", repo_root)


In [ ]:
# New non-Gaussian forward model + old Gaussian ILT inverse model (both 16x16)
fm_ng = ForwardModel3CNonGaussian(n_b=16, n_restrict_terms=500)
fm_ilt = create_forward_model(profile=16)

print("fm_ng n_b =", fm_ng.n_b)
print("fm_ilt n_b / n_d =", fm_ilt.n_b, fm_ilt.n_d)


In [ ]:
def run_ilt(signal_16x16, alpha=0.02, post_sharpen=True, sharpen_sigma=0.9, sharpen_strength=0.30):
    return fm_ilt.compute_ilt_nnls(
        signal_16x16,
        alpha=alpha,
        post_sharpen=post_sharpen,
        sharpen_sigma=sharpen_sigma,
        sharpen_strength=sharpen_strength,
        renorm=True,
    )


def diag_profile_stats(f):
    d = np.diag(np.asarray(f, dtype=np.float64))
    s = d.sum() + 1e-12
    p = d / s
    idx = np.arange(len(d), dtype=np.float64)
    mu = float((idx * p).sum())
    var = float(((idx - mu) ** 2 * p).sum())
    std = float(np.sqrt(max(var, 0.0)))
    peak_idx = int(np.argmax(d))

    thr = 0.20 * float(d.max() + 1e-12)
    n_local = 0
    for i in range(1, len(d) - 1):
        if d[i] >= d[i - 1] and d[i] >= d[i + 1] and d[i] >= thr:
            n_local += 1
    if len(d) >= 2:
        if d[0] >= d[1] and d[0] >= thr:
            n_local += 1
        if d[-1] >= d[-2] and d[-1] >= thr:
            n_local += 1

    return {
        "diag": d,
        "diag_std": std,
        "diag_peak_idx": peak_idx,
        "diag_local_peaks_above_20pct": int(n_local),
    }


def reprojection_rmse(signal_ref, f_est):
    signal_fit = fm_ilt.compute_signal(f_est, noise_sigma=0.0, normalize=True, noise_model=None)
    rmse = float(np.sqrt(np.mean((signal_fit - signal_ref) ** 2)))
    return rmse, signal_fit


def estimate_effective_diffusivity_from_kernel(b, k, n_points=6):
    mask = (b > 0) & (k > 0)
    x = np.asarray(b[mask], dtype=np.float64)
    y = -np.log(np.asarray(k[mask], dtype=np.float64))
    if x.size == 0:
        return 1e-12
    n = min(max(3, n_points), x.size)
    x = x[:n]
    y = y[:n]
    A = np.vstack([x, np.ones_like(x)]).T
    slope, _ = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(max(slope, 1e-12))


def nearest_d_index(d_value, d_grid):
    return int(np.argmin(np.abs(np.log(d_grid) - np.log(max(d_value, 1e-12)))))


def pathway_local_masses(f_est, idx_map, radius=1):
    f = np.asarray(f_est, dtype=np.float64)
    n = f.shape[0]
    comps = ["E", "T", "S"]
    masses = {}
    union_mask = np.zeros_like(f, dtype=bool)

    for ci in comps:
        for cj in comps:
            i = idx_map[ci]
            j = idx_map[cj]
            i0, i1 = max(0, i - radius), min(n, i + radius + 1)
            j0, j1 = max(0, j - radius), min(n, j + radius + 1)
            m = np.zeros_like(f, dtype=bool)
            m[i0:i1, j0:j1] = True
            union_mask |= m
            masses[ci + cj] = float(f[m].sum())

    local_total = float(sum(masses.values())) + 1e-12
    masses_norm = {k: float(v / local_total) for k, v in masses.items()}
    union_mass = float(f[union_mask].sum())
    return masses, masses_norm, union_mass


def compare_weight_vs_ilt_masses(details, f_est, D_E, D_I, l_T, R_S, radius=1):
    kern = fm_ng.compartment_kernels(
        g=fm_ng.G1,
        extracellular_diffusivity=D_E,
        intracellular_diffusivity=D_I,
        axon_restricted_length=l_T,
        sphere_radius=R_S,
    )
    d_eff = {
        "E": estimate_effective_diffusivity_from_kernel(fm_ng.b1, kern["E"]),
        "T": estimate_effective_diffusivity_from_kernel(fm_ng.b1, kern["T"]),
        "S": estimate_effective_diffusivity_from_kernel(fm_ng.b1, kern["S"]),
    }
    idx_map = {k: nearest_d_index(v, fm_ilt.D1) for k, v in d_eff.items()}

    m_raw, m_norm, in_union = pathway_local_masses(f_est, idx_map, radius=radius)
    w_true = details["pathway_weights"]

    diag_keys = ["EE", "TT", "SS"]
    true_diag = float(sum(w_true[k] for k in diag_keys))
    true_off = float(sum(w_true[k] for k in w_true if k not in diag_keys))
    ilt_diag = float(sum(m_norm[k] for k in diag_keys))
    ilt_off = float(sum(m_norm[k] for k in m_norm if k not in diag_keys))

    return {
        "d_eff": d_eff,
        "idx_map": idx_map,
        "w_true": dict(w_true),
        "ilt_masses_raw": m_raw,
        "ilt_masses_norm": m_norm,
        "ilt_union_mass": float(in_union),
        "diag_off_true": {"diag": true_diag, "off": true_off},
        "diag_off_ilt": {"diag": ilt_diag, "off": ilt_off},
    }


def run_nongaussian_case(phi, q, tm, D_E, D_I, l_T, R_S, noise_sigma=0.0, seed=0, alpha=0.02, post_sharpen=True):
    signal_clean, details = fm_ng.compute_signal(
        phi=phi,
        q=q,
        mixing_time=tm,
        extracellular_diffusivity=D_E,
        intracellular_diffusivity=D_I,
        axon_restricted_length=l_T,
        sphere_radius=R_S,
        normalize=True,
    )

    if noise_sigma > 0:
        rng = np.random.default_rng(seed)
        signal_in = fm_ng.add_rician_noise(signal_clean, noise_sigma=noise_sigma, normalize=True, rng=rng)
    else:
        signal_in = signal_clean.copy()

    f_est = run_ilt(signal_in, alpha=alpha, post_sharpen=post_sharpen)
    dei_true = fm_ng.compute_dei_from_weight_matrix(details["weight_matrix"])
    dei_est = compute_dei(f_est, diagonal_band_width=2)
    dei_bias = float(dei_est - dei_true)
    rmse, signal_fit = reprojection_rmse(signal_clean, f_est)

    return {
        "signal_clean": signal_clean,
        "signal_in": signal_in,
        "signal_fit": signal_fit,
        "f_est": f_est,
        "details": details,
        "dei_true": float(dei_true),
        "dei_est": float(dei_est),
        "dei_bias": dei_bias,
        "rmse": rmse,
    }


## 1) Gaussian clean signal → ILT
在 Gaussian 模型下，ILT 应该恢复出集中且较尖锐的峰（作为 pipeline 正常性的 sanity check）。


In [ ]:
# Gaussian clean signal generation on old model grid
mixing_time = 0.08
diffusions = np.array([0.7e-9, 1.7e-9, 2.8e-9], dtype=np.float64)
volume_fractions = np.array([0.45, 0.35, 0.20], dtype=np.float64)
exchange_rates = (3.0, 1.5, 0.8)

f_gt_g, s_clean_g, params_g = fm_ilt.generate_3c_validation_spectrum(
    diffusions=diffusions,
    volume_fractions=volume_fractions,
    exchange_rates=exchange_rates,
    mixing_time=mixing_time,
    jitter_pixels=0,
    smoothing_sigma=0.0,
    normalize=True,
)

alpha_g = 0.01
f_ilt_g = run_ilt(s_clean_g, alpha=alpha_g, post_sharpen=False)
st_gt = diag_profile_stats(f_gt_g)
st_ilt = diag_profile_stats(f_ilt_g)
rmse_g, _ = reprojection_rmse(s_clean_g, f_ilt_g)

print('Gaussian clean -> ILT summary')
print(f'alpha={alpha_g}, reproj RMSE={rmse_g:.6f}')
print('GT  diag peak idx/std/local_peaks:', st_gt['diag_peak_idx'], f"{st_gt['diag_std']:.3f}", st_gt['diag_local_peaks_above_20pct'])
print('ILT diag peak idx/std/local_peaks:', st_ilt['diag_peak_idx'], f"{st_ilt['diag_std']:.3f}", st_ilt['diag_local_peaks_above_20pct'])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
axes[0].imshow(s_clean_g, origin='lower', cmap='viridis', interpolation='nearest')
axes[0].set_title('Gaussian clean signal')
axes[0].set_xlabel('b2'); axes[0].set_ylabel('b1')

axes[1].imshow(f_gt_g, origin='lower', cmap='magma', interpolation='nearest')
axes[1].set_title('GT Gaussian spectrum')
axes[1].set_xlabel('D2'); axes[1].set_ylabel('D1')

axes[2].imshow(f_ilt_g, origin='lower', cmap='magma', interpolation='nearest')
axes[2].set_title('ILT spectrum (clean)')
axes[2].set_xlabel('D2'); axes[2].set_ylabel('D1')

axes[3].plot(st_gt['diag'], '-o', ms=3, label='GT diag')
axes[3].plot(st_ilt['diag'], '-o', ms=3, label='ILT diag')
axes[3].set_title('Diagonal profile')
axes[3].set_xlabel('diag index')
axes[3].legend()
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 2) Non-Gaussian clean signal，sweep sphere radius
观察 ILT 峰位和展宽随球半径变化，说明 ILT 给出的是 apparent diffusion 表征。


In [ ]:
phi_sphere_only = np.array([0.0, 0.0, 1.0], dtype=np.float64)
q_zero = np.zeros((3, 3), dtype=np.float64)

# shared physics
mixing_time = 0.08
D_E = 1.7e-9
D_I = 0.7e-9
l_T = 1.0e-6

Rs_um_list = [1, 2, 4, 6, 10, 20, 40]
alpha_clean_ng = 0.02

rows = []
viz_cases = []
for i, rs_um in enumerate(Rs_um_list):
    R_S = rs_um * 1e-6
    c = run_nongaussian_case(
        phi=phi_sphere_only, q=q_zero, tm=mixing_time,
        D_E=D_E, D_I=D_I, l_T=l_T, R_S=R_S,
        noise_sigma=0.0, seed=100 + i,
        alpha=alpha_clean_ng, post_sharpen=True,
    )
    st = diag_profile_stats(c['f_est'])

    kern = fm_ng.compartment_kernels(fm_ng.G1, D_E, D_I, l_T, R_S)
    d_eff_s = estimate_effective_diffusivity_from_kernel(fm_ng.b1, kern['S'])

    rows.append({
        'R_s_um': rs_um,
        'D_eff_S': d_eff_s,
        'diag_peak_idx': st['diag_peak_idx'],
        'diag_std': st['diag_std'],
        'diag_local_peaks': st['diag_local_peaks_above_20pct'],
        'rmse': c['rmse'],
    })

    if rs_um in [1, 6, 20, 40]:
        viz_cases.append((rs_um, c, st))

print('R_s(um) | D_eff_S(1e-9 m^2/s) | peak_idx | diag_std | n_local_peaks | rmse')
for r in rows:
    print(f"{r['R_s_um']:7.1f} | {r['D_eff_S']/1e-9:16.3f} | {r['diag_peak_idx']:8d} | {r['diag_std']:.3f} | {r['diag_local_peaks']:13d} | {r['rmse']:.5f}")

fig, axes = plt.subplots(len(viz_cases), 3, figsize=(10.5, 2.6 * len(viz_cases)))
if len(viz_cases) == 1:
    axes = np.array([axes])

for i, (rs_um, c, st) in enumerate(viz_cases):
    axes[i, 0].imshow(c['signal_clean'], origin='lower', cmap='viridis', interpolation='nearest')
    axes[i, 0].set_title(f'S clean (R_s={rs_um:.0f}um)')
    axes[i, 0].set_xlabel('b2'); axes[i, 0].set_ylabel('b1')

    axes[i, 1].imshow(c['f_est'], origin='lower', cmap='magma', interpolation='nearest')
    axes[i, 1].set_title('ILT spectrum')
    axes[i, 1].set_xlabel('D2'); axes[i, 1].set_ylabel('D1')

    axes[i, 2].plot(st['diag'], '-o', ms=3)
    axes[i, 2].set_title(f"Diag (std={st['diag_std']:.2f}, peak={st['diag_peak_idx']})")
    axes[i, 2].set_xlabel('diag index')
    axes[i, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Trend plot
rs = np.array([r['R_s_um'] for r in rows], dtype=np.float64)
peak_idx = np.array([r['diag_peak_idx'] for r in rows], dtype=np.float64)
diag_std = np.array([r['diag_std'] for r in rows], dtype=np.float64)

fig, ax1 = plt.subplots(1, 1, figsize=(6.8, 3.8))
ax1.plot(rs, peak_idx, '-o', label='diag peak index')
ax1.set_xlabel('sphere radius (um)')
ax1.set_ylabel('peak index')
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(rs, diag_std, '-s', color='tab:red', label='diag std')
ax2.set_ylabel('diag std', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='best')
plt.title('Non-Gaussian clean: peak shift / broadening vs sphere radius')
plt.tight_layout()
plt.show()


## 3) Non-Gaussian noisy signal，sweep SNR 和 alpha
用 DEI bias 和 pathway MAE 看噪声与正则化参数对 ILT 的影响。


In [ ]:
phi = np.array([0.45, 0.35, 0.20], dtype=np.float64)
q = fm_ng.build_generator(k_et=3.0, k_te=2.0, k_es=1.5, k_se=1.2, k_ts=0.0, k_st=0.0)

mixing_time = 0.08
D_E = 1.7e-9
D_I = 0.7e-9
l_T = 1.0e-6
R_S = 4.0e-6

noise_sigmas = np.array([0.0025, 0.005, 0.01, 0.02], dtype=np.float64)
alpha_list = np.array([0.005, 0.01, 0.02, 0.05], dtype=np.float64)
repeats = 12
path_keys = ['EE','ET','ES','TE','TT','TS','SE','ST','SS']

pathway_mae = np.zeros((len(noise_sigmas), len(alpha_list)), dtype=np.float64)
abs_dei_bias = np.zeros_like(pathway_mae)
rmse_map = np.zeros_like(pathway_mae)

for i, sigma in enumerate(noise_sigmas):
    for j, alpha in enumerate(alpha_list):
        path_err_runs = []
        dei_bias_runs = []
        rmse_runs = []

        for r in range(repeats):
            c = run_nongaussian_case(
                phi=phi, q=q, tm=mixing_time,
                D_E=D_E, D_I=D_I, l_T=l_T, R_S=R_S,
                noise_sigma=float(sigma),
                seed=20260000 + 1000*i + 100*j + r,
                alpha=float(alpha),
                post_sharpen=True,
            )
            cmp_res = compare_weight_vs_ilt_masses(c['details'], c['f_est'], D_E, D_I, l_T, R_S, radius=1)

            w_true = np.array([cmp_res['w_true'][k] for k in path_keys], dtype=np.float64)
            w_ilt = np.array([cmp_res['ilt_masses_norm'][k] for k in path_keys], dtype=np.float64)

            path_err_runs.append(float(np.mean(np.abs(w_true - w_ilt))))
            dei_bias_runs.append(abs(float(c['dei_bias'])))
            rmse_runs.append(float(c['rmse']))

        pathway_mae[i, j] = float(np.mean(path_err_runs))
        abs_dei_bias[i, j] = float(np.mean(dei_bias_runs))
        rmse_map[i, j] = float(np.mean(rmse_runs))

# Print best alpha per noise level
print('noise_sigma | approx SNR | best alpha (pathway MAE) | best pathway_MAE')
for i, sigma in enumerate(noise_sigmas):
    best_j = int(np.argmin(pathway_mae[i]))
    snr = 1.0 / sigma
    print(f"{sigma:10.4f} | {snr:10.1f} | {alpha_list[best_j]:>8.4f} | {pathway_mae[i, best_j]:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))

im0 = axes[0].imshow(pathway_mae, origin='lower', cmap='magma', aspect='auto')
axes[0].set_title('Mean pathway MAE')
axes[0].set_xticks(np.arange(len(alpha_list)))
axes[0].set_xticklabels([f"{a:.3f}" for a in alpha_list])
axes[0].set_yticks(np.arange(len(noise_sigmas)))
axes[0].set_yticklabels([f"{s:.4f} (SNR~{1/s:.0f})" for s in noise_sigmas])
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('noise sigma')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(abs_dei_bias, origin='lower', cmap='magma', aspect='auto')
axes[1].set_title('Mean |DEI bias|')
axes[1].set_xticks(np.arange(len(alpha_list)))
axes[1].set_xticklabels([f"{a:.3f}" for a in alpha_list])
axes[1].set_yticks(np.arange(len(noise_sigmas)))
axes[1].set_yticklabels([f"{s:.4f}" for s in noise_sigmas])
axes[1].set_xlabel('alpha')
axes[1].set_ylabel('noise sigma')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(rmse_map, origin='lower', cmap='magma', aspect='auto')
axes[2].set_title('Mean reprojection RMSE')
axes[2].set_xticks(np.arange(len(alpha_list)))
axes[2].set_xticklabels([f"{a:.3f}" for a in alpha_list])
axes[2].set_yticks(np.arange(len(noise_sigmas)))
axes[2].set_yticklabels([f"{s:.4f}" for s in noise_sigmas])
axes[2].set_xlabel('alpha')
axes[2].set_ylabel('noise sigma')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(10.5, 3.8))
for j, alpha in enumerate(alpha_list):
    axes2[0].plot(noise_sigmas, pathway_mae[:, j], '-o', label=f'alpha={alpha:.3f}')
    axes2[1].plot(noise_sigmas, abs_dei_bias[:, j], '-o', label=f'alpha={alpha:.3f}')

axes2[0].set_title('Pathway MAE vs noise')
axes2[0].set_xlabel('noise sigma')
axes2[0].set_ylabel('mean pathway MAE')
axes2[0].grid(alpha=0.3)

axes2[1].set_title('|DEI bias| vs noise')
axes2[1].set_xlabel('noise sigma')
axes2[1].set_ylabel('mean |DEI bias|')
axes2[1].grid(alpha=0.3)
axes2[1].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()



## 结果解读建议
- 实验1看 `ILT` 是否在 Gaussian clean 上恢复集中峰（验证流程）。
- 实验2看 `diag peak index` 与 `diag std` 随半径变化（apparent diffusion / broadening）。
- 实验3看噪声升高时 `pathway MAE`、`|DEI bias|` 是否整体变差，以及 alpha 是否存在稳健区间。
